*This code is a companion to the book **Mastering PyTorch and Lightning: A Step-by-Step Practical Guide with QA** by Aghiles Kebaili*
> This notebook contains the raw code for Chapter 12: Production Callbacks & Advanced Telemetry. It introduces checkpointing, early stopping, logger configuration, and the monitoring layer that sits outside the core model code.

A production pipeline is not only about optimization; it must also preserve the best model, stop safely on stagnation, and keep useful signals for debugging and comparison across runs.

## How Callbacks Extend the Trainer
### Step 0: Importing Previously Implemented Classes

In [ ]:
import os
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import pytorch_lightning as pl

# Callback logic stays separate from the model definition and monitors Trainer behavior from the outside.
class VisionDataModule(pl.LightningDataModule):
    def __init__(
        self,
        data_dir: str = "./data",
        batch_size: int = 256,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.data_dir = data_dir
        self.batch_size = batch_size
        self.num_workers = min(4, os.cpu_count() or 1)
        self.pin_memory = torch.cuda.is_available()

        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(
                (0.5, 0.5, 0.5),
                (0.5, 0.5, 0.5),
            ),
        ])

    def prepare_data(self):
        datasets.CIFAR10(
            self.data_dir, train=True, download=True
        )
        datasets.CIFAR10(
            self.data_dir, train=False, download=True
        )

    def setup(self, stage: str | None = None):
        if stage in ("fit", "validate", None):
            if not hasattr(self, "cifar_train"):
                full_dataset = datasets.CIFAR10(
                    self.data_dir,
                    train=True,
                    transform=self.transform,
                )
                generator = torch.Generator().manual_seed(42)
                self.cifar_train, self.cifar_val = random_split(
                    full_dataset,
                    [45000, 5000],
                    generator=generator,
                )

        if stage in ("test", "predict", None):
            if not hasattr(self, "cifar_test"):
                self.cifar_test = datasets.CIFAR10(
                    self.data_dir,
                    train=False,
                    transform=self.transform,
                )

    def _make_loader(self, dataset, shuffle):
        return DataLoader(
            dataset,
            batch_size=self.batch_size,
            shuffle=shuffle,
            num_workers=self.num_workers,
            pin_memory=self.pin_memory,
            persistent_workers=self.num_workers > 0,
        )

    def train_dataloader(self):
        return self._make_loader(self.cifar_train, shuffle=True)

    def val_dataloader(self):
        return self._make_loader(self.cifar_val, shuffle=False)

    def test_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)

    def predict_dataloader(self):
        return self._make_loader(self.cifar_test, shuffle=False)
    

class LitCIFARClassifier(pl.LightningModule):
    def __init__(self, lr: float = 1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.backbone(x)

    def _shared_step(self, batch, prefix):
        x, y = batch
        logits = self(x)
        loss = F.cross_entropy(logits, y)
        self.log(
            f"{prefix}_loss",
            loss,
            on_epoch=True,
            prog_bar=True,
            sync_dist=prefix != "train",
            batch_size=x.size(0),
        )
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def predict_step(self, batch, batch_idx):
        x, _ = batch
        return self(x).argmax(dim=1)

    def configure_optimizers(self):
        return torch.optim.AdamW(
            self.parameters(), lr=self.hparams.lr
        )

### Step 1: Protect Training Artifacts with ModelCheckpoint

In [ ]:
# ModelCheckpoint ranks and saves checkpoints based on a monitored validation metric.
# This separates model selection (best checkpoint) from fault recovery (last checkpoint).
from pytorch_lightning.callbacks import ModelCheckpoint

# save_top_k=1 keeps only the best checkpoint by val_loss, limiting disk space.
# save_last=True maintains the most recent complete training state for resumption.
# save_weights_only=False stores full training state (model + optimizer + scheduler + loop progress).
checkpoint = ModelCheckpoint(
    dirpath="./production_weights",
    filename="best-{epoch:02d}-{val_loss:.3f}",  # Embed epoch and loss in the filename
    monitor="val_loss",                          # Watch this metric
    mode="min",                                  # Lower loss is better
    save_top_k=1,                                # Retain only the single best checkpoint
    save_last=True,                              # Also save the most recent state
    save_weights_only=False,                     # Save complete training state for resumption
)

assert checkpoint.monitor == "val_loss"
assert checkpoint.save_top_k == 1


### Step 2: Control Compute Cost with EarlyStopping

In [ ]:
# EarlyStopping monitors a validation metric and stops training when it no longer improves.
# This prevents wasting compute on epochs that degrade generalization (overfitting).
# patience counts validation checks without sufficient improvement, not necessarily epochs.
from pytorch_lightning.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",            # Watch the validation loss
    patience=7,                    # Stop after 7 validation checks without improvement
    min_delta=0.005,               # Minimum decrease to count as improvement (overfitting protection)
    mode="min",                    # Seek lower loss
    check_finite=True,             # Also stop if loss becomes NaN or Inf (numerical safety)
    verbose=True,                  # Print stopping information
)

# Both callbacks monitor the same metric to stay synchronized.
assert early_stop.monitor == checkpoint.monitor


### Step 3: Track Optimization and Device Behavior

In [ ]:
# LearningRateMonitor watches how the optimizer's learning rate changes over time.
# This helps diagnose convergence issues and validate that schedulers behave as expected.
# DeviceStatsMonitor logs GPU/CPU memory and utilization for performance analysis.
from pytorch_lightning.callbacks import (
    DeviceStatsMonitor,
    LearningRateMonitor,
)

# Log learning rate at the end of each epoch (logging_interval="epoch").
# This provides coarser but less noisy data than step-level logging.
lr_monitor = LearningRateMonitor(logging_interval="epoch")

# Capture device statistics (memory, utilization) during training.
# These metrics help identify bottlenecks and memory leaks.
device_monitor = DeviceStatsMonitor()


## Logging Metrics and Hyperparameters

### Step 1: Configure Metric Aggregation

In [ ]:
# Metrics are logged via self.log() inside LightningModule hooks.
# The exact metric name "val_loss" must match what EarlyStopping and ModelCheckpoint monitor.
# Mismatched names leave callbacks without their required signal and training fails.
import torch.nn.functional as F

def validation_step(self, batch, batch_idx):
    inputs, targets = batch
    logits = self(inputs)
    loss = F.cross_entropy(logits, targets)

    # Log only at epoch level (not per-batch) to reduce noise.
    # sync_dist=True averages the metric across distributed processes for a global view.
    # batch_size helps weight the aggregation correctly across ranks.
    self.log(
        "val_loss",                  # Exact name monitored by callbacks
        loss,
        on_step=False,               # Don't log at batch level
        on_epoch=True,               # Aggregate and log once per epoch
        prog_bar=True,               # Show in progress bar
        sync_dist=True,              # Reduce across all processes for a global metric
        batch_size=inputs.size(0),   # Weight aggregation by batch size
    )

LitCIFARClassifier.validation_step = validation_step


### Step 2: Start with a Local Logger

In [ ]:
# CSVLogger provides a lightweight, dependency-free local logging backend.
# Metrics and hyperparameters are written to CSV files in the specified directory.
# This avoids remote dependencies and allows testing before integrating with cloud services.
from pytorch_lightning.loggers import CSVLogger

local_logger = CSVLogger(
    save_dir="./logs",              # Root directory for all logs
    name="cifar-monitoring",         # Experiment folder name
    version="run-01",               # Version subfolder for this run
)

# Verify that the logger was configured correctly.
assert "cifar-monitoring" in local_logger.log_dir


### Step 3: Assemble the Monitoring Configuration

In [17]:
# Complete production setup: model, data, callbacks, logger, and hardware all assembled.
# Callbacks act as independent observers: early_stop requests termination, checkpoint saves artifacts,
# and monitors (lr_monitor, device_monitor) record experimental metadata without changing model logic.
import pytorch_lightning as pl
import torch

model = LitCIFARClassifier(lr=1e-3)
datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=512,
)

use_cuda = torch.cuda.is_available()
precision = "16-mixed" if use_cuda else "32-true"

trainer = pl.Trainer(
    max_epochs=20,                  # Limit epochs (early_stop can terminate earlier)
    accelerator="auto",
    devices="auto",
    precision=precision,
    logger=local_logger,            # Attach CSV logger
    callbacks=[
        early_stop,                 # Stop on stagnation
        checkpoint,                 # Save best model and last state
        lr_monitor,                 # Track learning rate changes
        device_monitor,             # Track device utilization
    ],
    log_every_n_steps=20,           # Log metrics every 20 steps (reduces I/O overhead)
    enable_progress_bar=True,
)

# fit() runs the full training loop with all managed stages and callbacks invoked at proper times.
trainer.fit(model, datamodule=datamodule)

# After training, verify that both best and last checkpoints exist.
assert checkpoint.best_model_path    # Selected model for evaluation
assert checkpoint.last_model_path    # Latest state for resumption


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\Users\aghil\miniconda3\envs\work\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory G:\work\ebooks\mastering-pytorch-lightning-book\pytorch-core-foundation\production_weights exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved. New best score: 2.189


...

Metric val_loss improved by 0.011 >= min_delta = 0.005. New best score: 1.836


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.013 >= min_delta = 0.005. New best score: 1.823


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.011 >= min_delta = 0.005. New best score: 1.813
`Trainer.fit` stopped: `max_epochs=20` reached.


### Step 4: Resume from the Latest Complete State

In [18]:
# Resumption requires rebuilding the same model and DataModule, configuring callbacks and logger,
# and then passing the path to the last checkpoint (which contains optimizer and scheduler state).
# This allows optimization to continue from exactly where it left off, not from scratch.
resumed_model = LitCIFARClassifier(lr=1e-3)
resumed_datamodule = VisionDataModule(
    data_dir="./data",
    batch_size=512,
)

resume_trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices="auto",
    precision=precision,
    logger=local_logger,
    callbacks=[
        early_stop,
        checkpoint,
        lr_monitor,
        device_monitor,
    ],
)

# ckpt_path="./production_weights/last.ckpt" points to the complete training checkpoint.
# The Trainer will restore the model state and continue optimization from the saved global_step.
resume_trainer.fit(
    resumed_model,
    datamodule=resumed_datamodule,
    ckpt_path="./production_weights/last.ckpt",  # Resume from last complete state
)


Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Restoring states from the checkpoint path at ./production_weights/last.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type       | Params | Mode 
------------------------------------------------
0 | backbone | Sequential | 1.2 K  | train
------------------------------------------------
1.2 K     Trainable params
0         Non-trainable params
1.2 K     Total params
0.005     Total estimated model params size (MB)
6         Modules in train mode
0         Modules in eval mode
Restored all states from the checkpoint at ./production_weights/last.ckpt


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.009 >= min_delta = 0.005. New best score: 1.811


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.008 >= min_delta = 0.005. New best score: 1.803


Validation: |          | 0/? [00:00<?, ?it/s]

Metric val_loss improved by 0.008 >= min_delta = 0.005. New best score: 1.795
`Trainer.fit` stopped: `max_epochs=20` reached.
